# Cohort exploration — Phase 4

Cross-pool diagnostics on top of the 6 cohort parquets in `data/cohorts/`.
Run `scripts/build_cohorts.py` first if those parquets aren't present.

**Reading guide:**
- `mkt_total_pnl`, `mkt_profit_factor`, `mkt_dollar_win_rate` — primary cohort metrics, trust
- `mkt_sharpe` — diagnostic only; n_pos / active_days shown alongside
- `est_bankroll_usd_30d_max_approx` — descriptive (lifetime peak), do NOT use for sizing

## Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_infra.trader_profile import profile_trader, POOL_NAMES

TRADERS = ROOT / 'data' / 'traders.parquet'
COHORT_DIR = ROOT / 'data' / 'cohorts'
CLOSED_POS = ROOT / 'data' / 'closed_positions.parquet'

con = duckdb.connect()
con.execute('PRAGMA threads=4')
con.execute(f"CREATE VIEW traders AS SELECT * FROM read_parquet('{TRADERS}')")
con.execute("CREATE VIEW traders_filtered AS SELECT * FROM traders WHERE NOT is_operator_like")

# Load each cohort as its own view
for pool in POOL_NAMES:
    path = COHORT_DIR / f'{pool}.parquet'
    con.execute(f"CREATE OR REPLACE VIEW pool_{pool} AS SELECT * FROM read_parquet('{path}')")
    n = con.sql(f'SELECT count(*) FROM pool_{pool}').fetchone()[0]
    print(f'  pool_{pool:35s} → {n:>6,} rows')

## Pool size sensitivity

How much does each pool's size move if we shift the primary threshold by ±20%? Stable pools = safer cohorts; knife-edge pools = threshold-dependent.

In [ ]:
# (metric_name, base_threshold, pool_filter_excluding_that_metric)
SENSITIVITY = {
    'high_sharpe_directional': ('mkt_sharpe', 1.5,
        "mkt_sharpe < 10 AND n_closed_positions > 200 AND active_days > 90 AND negrisk_volume_share < 0.5 AND NOT is_operator_like"),
    'high_profit_factor_with_size': ('mkt_profit_factor', 2.0,
        "mkt_profit_factor < 100 AND mkt_total_pnl > 50000 AND n_closed_positions > 100 AND active_days > 90 AND NOT is_operator_like"),
    'negrisk_specialists': ('negrisk_volume_share', 0.7,
        "mkt_total_pnl > 50000 AND n_closed_positions > 200 AND active_days > 90 AND mkt_profit_factor > 1.3 AND NOT is_operator_like"),
    'patient_accumulators': ('style_avg_holding_hours', 168,
        "style_role_balance > 0.7 AND mkt_total_pnl > 100000 AND n_closed_positions > 100 AND active_days > 180 AND NOT is_operator_like"),
    'high_kelly_edge': ('mkt_kelly_fraction', 0.05,
        "mkt_kelly_fraction < 0.5 AND n_closed_positions > 200 AND active_days > 90 AND mkt_dollar_win_rate > 0.55 AND NOT is_operator_like"),
}

rows = []
for pool, (metric, base, other_filters) in SENSITIVITY.items():
    counts = {}
    for delta_label, mult in [('-20%', 0.8), ('base', 1.0), ('+20%', 1.2)]:
        t = base * mult
        sql = f"SELECT count(*) FROM traders WHERE {metric} > {t} AND {other_filters}"
        counts[delta_label] = con.sql(sql).fetchone()[0]
    rows.append({'pool': pool, 'metric': metric, 'base': base, **counts})

df = pd.DataFrame(rows)
df['ratio_-20_to_+20'] = df['-20%'] / df['+20%']
df

## Cohort overlap matrix

How many addresses appear in both pools (i, j)? Heatmap with annotations.

In [ ]:
matrix = np.zeros((len(POOL_NAMES), len(POOL_NAMES)), dtype=int)
for i, p1 in enumerate(POOL_NAMES):
    for j, p2 in enumerate(POOL_NAMES):
        n = con.sql(f"""
            SELECT count(*) FROM (
                SELECT address FROM pool_{p1}
                INTERSECT
                SELECT address FROM pool_{p2}
            )
        """).fetchone()[0]
        matrix[i, j] = n

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(matrix, cmap='viridis')
ax.set_xticks(range(len(POOL_NAMES)))
ax.set_yticks(range(len(POOL_NAMES)))
ax.set_xticklabels(POOL_NAMES, rotation=45, ha='right')
ax.set_yticklabels(POOL_NAMES)
for i in range(len(POOL_NAMES)):
    for j in range(len(POOL_NAMES)):
        c = 'white' if matrix[i,j] < matrix.max() * 0.5 else 'black'
        ax.text(j, i, str(matrix[i,j]), ha='center', va='center', color=c, fontsize=9)
ax.set_title('Cohort overlap (# addresses qualifying for both)')
fig.colorbar(im)
fig.tight_layout()

## Multi-pool members (3+ pools)

Most robust candidates — qualify by multiple independent metrics.

In [ ]:
paths = ', '.join(f"'{COHORT_DIR / (p + '.parquet')}'" for p in POOL_NAMES)
con.execute(f"""
    CREATE OR REPLACE TABLE _all_cohorts AS
    SELECT * FROM read_parquet([{paths}], filename = TRUE)
""")

multi = con.sql("""
    WITH per_addr AS (
        SELECT address, list(DISTINCT regexp_extract(filename, '([^/]+)\\.parquet$', 1)) AS pools,
               count(DISTINCT filename) AS n_pools
        FROM _all_cohorts GROUP BY address
    )
    SELECT address, n_pools, pools FROM per_addr WHERE n_pools >= 3
    ORDER BY n_pools DESC
""").fetchdf()

multi_df = con.sql("""
    WITH per_addr AS (
        SELECT address, count(DISTINCT filename) AS n_pools
        FROM _all_cohorts GROUP BY address
    )
    SELECT t.address, p.n_pools,
           round(t.mkt_total_pnl, 0) AS mkt_pnl,
           round(t.mkt_profit_factor, 2) AS pf,
           round(t.mkt_dollar_win_rate, 3) AS dwr,
           t.n_closed_positions AS n_pos, t.active_days,
           round(t.phantom_position_score, 2) AS phantom,
           round(t.negrisk_volume_share, 2) AS nr_share,
           round(t.style_role_balance, 2) AS role_bal
    FROM traders t
    JOIN per_addr p USING (address)
    WHERE p.n_pools >= 3
    ORDER BY t.mkt_total_pnl DESC NULLS LAST
""").fetchdf()
print(f'{len(multi_df)} addresses qualifying for 3+ pools')
multi_df.head(30)

## Top-20 union (across all pools)

Sorted by `mkt_total_pnl` across the union of all qualifying addresses. Includes Sharpe but **italicised — diagnostic only**, alongside `n_pos` and `active_days` so you can judge trust.

In [ ]:
top20 = con.sql("""
    WITH per_addr AS (
        SELECT address, count(DISTINCT filename) AS n_pools_qualified
        FROM _all_cohorts GROUP BY address
    )
    SELECT t.address, p.n_pools_qualified,
           round(t.mkt_total_pnl, 0) AS mkt_pnl,
           round(t.mkt_profit_factor, 2) AS pf,
           round(t.mkt_dollar_win_rate, 3) AS dwr,
           round(t.mkt_sharpe, 2) AS mkt_sharpe_diag,
           t.n_closed_positions AS n_pos,
           t.active_days,
           round(t.phantom_position_score, 2) AS phantom,
           round(t.negrisk_volume_share, 2) AS nr_share,
           round(t.style_role_balance, 2) AS role_bal,
           round(t.est_bankroll_usd_30d_max_approx, 0) AS lifetime_peak_bankroll
    FROM traders t
    JOIN per_addr p USING (address)
    ORDER BY t.mkt_total_pnl DESC NULLS LAST
    LIMIT 20
""").fetchdf()
top20

## Metric correlation matrix

Across the union of qualifying addresses. Metrics with correlation near 1 are redundant; near 0 means independent.

In [ ]:
union_df = con.sql("""
    SELECT DISTINCT t.mkt_profit_factor, t.mkt_dollar_win_rate, t.mkt_kelly_fraction,
                    t.mkt_total_pnl, t.n_closed_positions, t.mkt_sharpe
    FROM traders t JOIN _all_cohorts a USING (address)
    WHERE t.mkt_profit_factor IS NOT NULL
""").fetchdf()
corr = union_df.corr(method='pearson')
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_yticks(range(len(corr)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f'{corr.iat[i,j]:.2f}', ha='center', va='center', fontsize=9,
                color='white' if abs(corr.iat[i,j]) > 0.5 else 'black')
ax.set_title('Pearson correlation across qualifying addresses')
fig.colorbar(im); fig.tight_layout()
corr

## Style cluster scatter

All qualifying addresses in (style_role_balance, style_avg_holding_hours) space. Y log-scaled. Colour by mkt_total_pnl (log scale). Marker shape per pool.

In [ ]:
MARKERS = {'high_sharpe_directional':'o', 'high_profit_factor_with_size':'s',
           'negrisk_specialists':'^', 'sports_directional_fast':'D',
           'patient_accumulators':'P', 'high_kelly_edge':'X'}
fig, ax = plt.subplots(figsize=(11, 7))
for pool, marker in MARKERS.items():
    df = con.sql(f"""
        SELECT style_role_balance, style_avg_holding_hours, mkt_total_pnl
        FROM pool_{pool}
        WHERE style_role_balance IS NOT NULL
          AND style_avg_holding_hours > 0
          AND mkt_total_pnl > 0
    """).fetchdf()
    if df.empty: continue
    sc = ax.scatter(df['style_role_balance'], df['style_avg_holding_hours'],
                    c=np.log10(df['mkt_total_pnl']), cmap='plasma',
                    marker=marker, s=20, alpha=0.6, label=pool)
ax.set_yscale('log')
ax.set_xlabel('style_role_balance (1 = pure maker, 0 = pure taker)')
ax.set_ylabel('style_avg_holding_hours (log)')
ax.set_title('Style space — colour = log10(mkt_total_pnl)')
ax.legend(loc='best', fontsize=8)
fig.colorbar(sc, label='log10(mkt_total_pnl)')
fig.tight_layout()

## NegRisk specialist diagnostic

For Pool C members: split into (a) those with profitable mkt PnL on non-NegRisk markets too, (b) NegRisk-only profits.

In [ ]:
split = con.sql(f"""
    WITH per_split AS (
        SELECT cp.address,
               sum(CASE WHEN cp.neg_risk THEN cp.realised_pnl ELSE 0 END) AS negrisk_pnl,
               sum(CASE WHEN NOT cp.neg_risk THEN cp.realised_pnl ELSE 0 END) AS regular_pnl
        FROM read_parquet('{CLOSED_POS}') cp
        WHERE cp.address IN (SELECT address FROM pool_negrisk_specialists)
        GROUP BY cp.address
    )
    SELECT
        CASE WHEN regular_pnl > 0 THEN 'broad_edge' ELSE 'negrisk_only' END AS subset,
        count(*) AS n,
        round(avg(negrisk_pnl), 0) AS avg_negrisk_pnl,
        round(avg(regular_pnl), 0) AS avg_regular_pnl,
        round(median(negrisk_pnl), 0) AS med_negrisk_pnl,
        round(median(regular_pnl), 0) AS med_regular_pnl
    FROM per_split GROUP BY 1
""").fetchdf()
split

## Sharpe trustworthiness

Plot Sharpe vs n_closed_positions (X log-scale). Cluster of high-Sharpe at low n_pos = fluke region. Distributed across n_pos = real edge candidates.

In [ ]:
df = con.sql("""
    SELECT DISTINCT t.mkt_sharpe, t.n_closed_positions, t.active_days, t.mkt_total_pnl
    FROM traders t JOIN _all_cohorts a USING (address)
    WHERE t.mkt_sharpe IS NOT NULL AND t.n_closed_positions > 0
""").fetchdf()
fig, ax = plt.subplots(figsize=(11, 6))
sc = ax.scatter(df['n_closed_positions'], df['mkt_sharpe'],
                c=df['active_days'], cmap='viridis', s=15, alpha=0.6)
ax.set_xscale('log')
ax.set_xlabel('n_closed_positions (log)')
ax.set_ylabel('mkt_sharpe (annualised, naive)')
ax.axhline(5, color='red', ls='--', alpha=0.4, label='Sharpe = 5 (suspicious)')
ax.set_title('Sharpe vs sample size — colour = active_days')
ax.legend()
fig.colorbar(sc, label='active_days')
fig.tight_layout()

## Pool-by-pool detail

Per pool: top 3 by mkt_total_pnl + bottom 3 (the marginal qualifiers).

In [ ]:
for pool in POOL_NAMES:
    n = con.sql(f'SELECT count(*) FROM pool_{pool}').fetchone()[0]
    print(f'\n=== {pool} (n={n}) ===')
    cols = ("address, n_closed_positions AS n_pos, active_days, "
            "round(mkt_total_pnl, 0) AS mkt_pnl, round(mkt_profit_factor, 2) AS pf, "
            "round(mkt_dollar_win_rate, 3) AS dwr, round(mkt_sharpe, 2) AS mkt_sharpe_diag, "
            "round(phantom_position_score, 2) AS phantom")
    top = con.sql(f'SELECT {cols} FROM pool_{pool} ORDER BY mkt_total_pnl DESC LIMIT 3').fetchdf()
    bot = con.sql(f'SELECT {cols} FROM pool_{pool} ORDER BY mkt_total_pnl ASC LIMIT 3').fetchdf()
    print('top 3:'); print(top.to_string(index=False))
    print('marginal 3:'); print(bot.to_string(index=False))

## Trader profile — domah

Sanity check on the known leader.

In [ ]:
DOMAH = '0x9d84ce0306f8551e02efef1680475fc0f1dc1344'
p = profile_trader(DOMAH)
print('header:', p['header'])
print('\nheadline metrics:')
for k, v in p['headline_metrics'].items():
    print(f'  {k}: {v}')
print('\npools qualified for & cohort positioning:')
for pool, vals in p['cohort_positioning'].items():
    print(f'  {pool}: pnl_pct={vals["pnl_percentile"]:.0f}, pf_pct={vals["pf_percentile"]:.0f}')

In [ ]:
# Domah cumulative PnL chart
pm = p['pnl_monthly']
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(pm['month'], pm['cum_pnl'], marker='.', lw=1)
ax.axhline(0, color='k', alpha=0.3)
ax.set_title(f'domah — cumulative mkt_total_pnl by month')
ax.set_ylabel('cumulative PnL (USD)')
fig.tight_layout()

## Top 5 from cross-pool top-20

In [ ]:
top5_addrs = top20['address'].head(5).tolist()
for addr in top5_addrs:
    print(f'\n========== {addr} ==========')
    p = profile_trader(addr)
    print('verdict:', p['header']['verdict'])
    for k, v in list(p['headline_metrics'].items())[:8]:
        print(f'  {k}: {v}')
    print('  pools:', p['header']['pools_qualified'])

## One Pool E patient_accumulator + one Pool C negrisk_specialist

In [ ]:
pool_e_top = con.sql('SELECT address FROM pool_patient_accumulators ORDER BY mkt_total_pnl DESC LIMIT 1').fetchone()[0]
pool_c_top = con.sql('SELECT address FROM pool_negrisk_specialists ORDER BY mkt_total_pnl DESC LIMIT 1').fetchone()[0]
for label, addr in [('Pool E top patient_accumulator', pool_e_top), ('Pool C top negrisk_specialist', pool_c_top)]:
    print(f'\n========== {label}: {addr} ==========')
    p = profile_trader(addr)
    print('verdict:', p['header']['verdict'])
    for k, v in p['headline_metrics'].items():
        print(f'  {k}: {v}')
    print('  pools:', p['header']['pools_qualified'])